# 12 — Matching sensitivity: are the matched flows really the same flows?

Answers the robustness question with three analyses:

1. **Key variants** — directed vs canonical endpoints; match counts and
   label-disagreement under each.
2. **Duration agreement** — on canonically matched pairs, compare flow
   duration across releases. If matches are genuine, durations should agree
   closely except where the corrected extractor's timeout fix applies.
3. **Disagreement stability** — label-disagreement rate inside the
   duration-agreeing subset vs all matched flows. If the 4.40% were join
   artifacts, it would concentrate in duration-DISagreeing pairs; if it is
   labelling, the rate is stable.

Rebuilds keys from the cached parquets using the saved calibration.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np, json, time

orig = pd.read_parquet(os.path.join(C.INTERIM, 'original.parquet'))
impr = pd.read_parquet(os.path.join(C.INTERIM, 'improved.parquet'))
cal = json.load(open(os.path.join(C.RESULTS, 'match_calibration.json')))
print(len(orig), len(impr), '| shift', cal.get('shift_applied_hours', 3), 'h')

Mounted at /content/drive
2830743 2099976 | shift 3 h


In [2]:
# Parse timestamps exactly as notebook 03 did (per-element, 12h repair, +3h)
def parse_side(df, is_original):
    ts = pd.to_datetime(df['timestamp'], errors='coerce', dayfirst=True,
                        format='mixed')
    if float((ts.dt.month == 7).mean()) < 0.99:
        ts2 = pd.to_datetime(df['timestamp'], errors='coerce', dayfirst=False,
                             format='mixed')
        if float((ts2.dt.month == 7).mean()) > float((ts.dt.month == 7).mean()):
            ts = ts2
    if is_original:
        h = ts.dt.hour
        fix = h.between(1, 7)
        ts = ts + pd.to_timedelta(fix.astype('int64') * 12, unit='h')
        ts = ts + pd.Timedelta(hours=cal.get('shift_applied_hours', 3))
    print('parsed: month==7', f"{(ts.dt.month == 7).mean():.1%}",
          'NaT', f"{ts.isna().mean():.2%}")
    return ts

ts_o = parse_side(orig, True)
ts_i = parse_side(impr, False)

parsed: month==7 100.0% NaT 0.00%
parsed: month==7 100.0% NaT 0.00%


In [3]:
def endpoints(df):
    a = df['src_ip'].astype(str).str.strip() + ':' + \
        pd.to_numeric(df['src_port'], errors='coerce').astype('Int64').astype(str)
    b = df['dst_ip'].astype(str).str.strip() + ':' + \
        pd.to_numeric(df['dst_port'], errors='coerce').astype('Int64').astype(str)
    pr = pd.to_numeric(df['protocol'], errors='coerce').astype('Int64').astype(str)
    return a.values, b.values, pr.values

def keys(df, ts, directed):
    a, b, pr = endpoints(df)
    if directed:
        k1, k2 = a, b
    else:
        first = a <= b
        k1, k2 = np.where(first, a, b), np.where(first, b, a)
    k = pd.DataFrame({'k1': k1, 'k2': k2, 'pr': pr}, index=df.index)
    k['mn'] = ts.dt.floor('min').astype(str)
    return k

# lighter implementation: merge frames directly
def dur_col(df, tag):
    cands = [c for c in df.columns if 'duration' in c.lower()]
    print(tag, 'duration column:', cands[0] if cands else 'NONE')
    return cands[0] if cands else None

DUR_O, DUR_I = dur_col(orig, 'original'), dur_col(impr, 'improved')

def match_full(directed):
    KEY = ['k1', 'k2', 'pr', 'mn']
    ko = keys(orig, ts_o, directed);  ko['label_o'] = orig['label'].values
    ko['dur_o'] = (pd.to_numeric(orig[DUR_O], errors='coerce').values
                   if DUR_O else np.nan)
    ki = keys(impr, ts_i, directed);  ki['label_i'] = impr['label'].values
    ki['dur_i'] = (pd.to_numeric(impr[DUR_I], errors='coerce').values
                   if DUR_I else np.nan)
    def uniq(k):
        return k[~k.duplicated(subset=KEY, keep=False)]
    ko, ki = uniq(ko.dropna(subset=['mn'])), uniq(ki.dropna(subset=['mn']))
    m = ko.merge(ki, on=KEY, how='inner')
    fo, fi = H.coarse_class(m['label_o']), H.coarse_class(m['label_i'])
    dis = (fo != fi)
    b2a = ((fo == 'BENIGN') & (fi != 'BENIGN')).sum()
    return {'variant': 'directed' if directed else 'canonical',
            'unique_original': len(ko), 'unique_improved': len(ki),
            'matched': len(m),
            'pct_of_unique_original': round(100 * len(m) / len(ko), 2),
            'family_disagree_pct': round(100 * dis.mean(), 4),
            'benign_to_attack': int(b2a)}, m

r_can, m_can = match_full(directed=False)
r_dir, m_dir = match_full(directed=True)
sens = pd.DataFrame([r_can, r_dir])
H.save_table(sens, 't12_key_sensitivity.csv')
sens

original duration column: flow duration
improved duration column: flow duration
saved /content/drive/MyDrive/research/ids-label-correction/results/t12_key_sensitivity.csv (2, 7)


,variant,unique_original,unique_improved,matched,pct_of_unique_original,family_disagree_pct,benign_to_attack
0,canonical,1767843,2073720,1604995,90.79,4.0505,65001
1,directed,2218218,2074039,1797981,81.06,3.1762,57095


In [4]:
if m_can['dur_o'].notna().sum() == 0 or m_can['dur_i'].notna().sum() == 0:
    print('duration column missing on one side - stability analysis skipped;')
    print('key-variant sensitivity (previous cell) stands on its own')
else:
    # Duration agreement on canonical matches + disagreement stability
    m = m_can.dropna(subset=['dur_o', 'dur_i']).copy()
    m['dur_diff_s'] = (m['dur_o'] - m['dur_i']).abs() / 1e6   # microseconds -> s
    fo, fi = H.coarse_class(m['label_o']), H.coarse_class(m['label_i'])
    m['family_disagree'] = (fo != fi)

    qs = m['dur_diff_s'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(3)
    print('|duration_orig - duration_impr| quantiles (s):')
    print(qs.to_string())

    rows = []
    for thr in [0.1, 1.0, 5.0, np.inf]:
        sub = m[m['dur_diff_s'] <= thr]
        rows.append({'duration_tolerance_s': thr,
                     'pairs': len(sub),
                     'pct_of_matched': round(100 * len(sub) / len(m), 2),
                     'family_disagree_pct': round(100 * sub['family_disagree'].mean(), 4)})
    stab = pd.DataFrame(rows)
    H.save_table(stab, 't12_duration_stability.csv')
    print('\nIf 4.40% were join artifacts it would concentrate at large duration')
    print('differences; a stable rate across tolerances supports genuine matches:')
    stab

|duration_orig - duration_impr| quantiles (s):
0.50     0.000
0.75     0.000
0.90     0.051
0.95    52.262
0.99    99.549
saved /content/drive/MyDrive/research/ids-label-correction/results/t12_duration_stability.csv (4, 4)

If 4.40% were join artifacts it would concentrate at large duration
differences; a stable rate across tolerances supports genuine matches:
